In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from tqdm import tqdm
from copy import deepcopy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.metrics import classification_report, f1_score, confusion_matrix, multilabel_confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split

In [3]:
PATH_DATA = '../data'

In [4]:
train = pd.read_csv(f'{PATH_DATA}/train.csv')
test = pd.read_csv(f'{PATH_DATA}/test.csv')

In [5]:
train = train.drop(columns=['ID'])
train = train.reset_index(drop=True)
train.dropna(axis=1, how='all', inplace=True)

test = test.drop(columns=['ID'])
test = test.reset_index(drop=True)
test.dropna(axis=1, how='all', inplace=True)

In [6]:
train['TITLE'] = train['TITLE'].fillna('')
train['ABSTRACT'] = train['ABSTRACT'].fillna('')

test['TITLE'] = test['TITLE'].fillna('')
test['ABSTRACT'] = test['ABSTRACT'].fillna('')

In [7]:
train['text'] = train['TITLE'] + ' ' + train['ABSTRACT']
test['text'] = test['TITLE'] + ' ' + test['ABSTRACT']

In [8]:
train.drop(columns=['TITLE', 'ABSTRACT'], inplace=True)
test.drop(columns=['TITLE', 'ABSTRACT'], inplace=True)

In [9]:
train.isna().sum()

Computer Science        0
Physics                 0
Mathematics             0
Statistics              0
Quantitative Biology    0
Quantitative Finance    0
text                    0
dtype: int64

In [10]:
train.head()

,Computer Science,Physics,Mathematics,Statistics,Quantitative Biology,Quantitative Finance,text
0,1,0,0,0,0,0,Reconstructing Subject-Specific Effect Maps ...
1,1,0,0,0,0,0,Rotation Invariance Neural Network Rotation ...
2,0,0,1,0,0,0,Spherical polyharmonics and Poisson kernels fo...
3,0,0,1,0,0,0,A finite element approximation for the stochas...
4,1,0,0,1,0,0,Comparative study of Discrete Wavelet Transfor...


In [11]:
vectorizer = TfidfVectorizer(
    max_features=10_000,  # Keep top 10k words (adjust as needed)
    stop_words='english',  # Remove common words ("the", "a", etc.)
    ngram_range=(1, 2)    # Include 1-word and 2-word phrases
)

In [12]:
X_train = vectorizer.fit_transform(train['text'])
X_test = vectorizer.transform(test['text'])

In [13]:
print('train shape:', X_train.shape)
print('test shape:', X_test.shape)

train shape: (20972, 10000)
test shape: (8989, 10000)


In [14]:
label_cols = ['Computer Science', 'Physics', 'Mathematics',
    'Statistics', 'Quantitative Biology', 'Quantitative Finance']
y_train = train[label_cols]

In [15]:
X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train, \
                                                            test_size = 0.3, \
                                                            random_state = 42)

### Classificador em Cadeia Básico

In [16]:
base_clf = LogisticRegression(class_weight='balanced', max_iter=1000)
model_chain = ClassifierChain(base_clf)

In [17]:
model_chain.fit(X_train_split, y_train_split)
y_val_pred_chain = model_chain.predict(X_val)

In [18]:
print(classification_report(y_val, y_val_pred_chain, target_names = label_cols))

                      precision    recall  f1-score   support

    Computer Science       0.80      0.89      0.84      2535
             Physics       0.86      0.88      0.87      1849
         Mathematics       0.79      0.81      0.80      1711
          Statistics       0.73      0.83      0.78      1554
Quantitative Biology       0.49      0.18      0.26       194
Quantitative Finance       0.77      0.25      0.38        67

           micro avg       0.79      0.84      0.81      7910
           macro avg       0.74      0.64      0.66      7910
        weighted avg       0.79      0.84      0.81      7910
         samples avg       0.83      0.86      0.83      7910



### Aplicando Algoritmo Genético para encontrar a melhor ordem de classes

In [19]:
# Genetic Algorithm Parameters
POPULATION_SIZE = 20
GENERATIONS = 50
MUTATION_RATE = 0.2
ELITISM = 2

In [20]:
def initialize_population(num_classes):
    """
    Inicializa a população com permutações aleatórias dos índices das classes.

    Args:
        num_classes (int): Número de classes.

    Retorna:
        list: Lista contendo permutações aleatórias dos índices das classes.
    """
    return [np.random.permutation(num_classes).tolist() for _ in range(POPULATION_SIZE)]

In [21]:
def evaluate_individual(individual, X_train, y_train, X_val, y_val):
    """
    Avalia uma ordem de classes treinando uma cadeia de classificadores.

    Args:
        individual (list): Ordem das classes a ser avaliada.
        X_train (sparse matrix): Dados de treinamento.
        y_train (DataFrame): Rótulos de treinamento.
        X_val (sparse matrix): Dados de validação.
        y_val (DataFrame): Rótulos de validação.

    Retorna:
        float: Pontuação F1 (média micro) para a ordem avaliada.
    """
    try:
        base_clf = LogisticRegression(class_weight='balanced', max_iter=500)
        model = ClassifierChain(base_clf, order=individual)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        return f1_score(y_val, y_pred, average='micro')
    except:
        return 0.0  # Retorna a pontuação mínima se ocorrer um erro

In [22]:
def selection(population, fitness_scores):
    """
    Seleção por torneio.

    Args:
        population (list): População atual, onde cada indivíduo é uma permutação de classes.
        fitness_scores (list): Lista de pontuações de aptidão correspondentes a cada indivíduo na população.

    Retorna:
        list: Um indivíduo selecionado da população com base na pontuação de aptidão.
    """
    idx = np.random.choice(len(population), size=2, replace=False)
    return deepcopy(population[idx[0]] if fitness_scores[idx[0]] > fitness_scores[idx[1]] else population[idx[1]])

In [23]:
def crossover(parent1, parent2):
    """
    Crossover ordenado (OX) para permutações.

    Este método cria um novo indivíduo (filho) combinando partes de dois pais.
    Uma subseção do primeiro pai é copiada diretamente para o filho, e os valores
    restantes são preenchidos na ordem em que aparecem no segundo pai, sem duplicatas.

    Args:
        parent1 (list): O primeiro pai, representado como uma lista de índices de classes.
        parent2 (list): O segundo pai, representado como uma lista de índices de classes.

    Retorna:
        list: Um novo indivíduo (filho) gerado a partir dos dois pais.
    """
    size = len(parent1)
    start, end = sorted(np.random.choice(range(size), size=2, replace=False))
    child = [None] * size
    child[start:end] = parent1[start:end]
    
    current_pos = end % size
    for val in parent2:
        if val not in child:
            child[current_pos] = val
            current_pos = (current_pos + 1) % size
    
    return child

In [24]:
def mutate(individual):
    """
    Mutação por troca.

    Este método realiza uma mutação em um indivíduo (permutação de classes) 
    trocando aleatoriamente a posição de dois elementos, com base em uma 
    taxa de mutação predefinida.

    Args:
        individual (list): Um indivíduo representado como uma lista de índices de classes.

    Retorna:
        list: O indivíduo possivelmente mutado.
    """
    if np.random.random() < MUTATION_RATE:
        idx = np.random.choice(len(individual), size=2, replace=False)
        individual[idx[0]], individual[idx[1]] = individual[idx[1]], individual[idx[0]]
    return individual

### Função Objetivo

O problema consiste em determinar a melhor ordenação $\mathbf{O} = [o_1, o_2, \dots, o_n]$ das classes em uma cadeia de classificadores, de modo a **maximizar a pontuação $F1_{\text{micro}}$**. Formalmente, a função objetivo é definida como:

$$
\max_{\mathbf{O} \in S_n} F1_{\text{micro}}(\mathbf{O})
$$

onde:
- $n$ representa o número de classes, para o problema especifico $n = 6$;
- $\mathbf{O}$ representa uma permutação **simples**, ou seja sem repetição, das $n$ classes;
- $S_n$ é o conjunto de todas as permutações possíveis de $n$ elementos. O número total de permutações possíveis de um conjunto de $n$ classes é dado por $n!$, logo $n! = 720$.
- $|S_i| = n$ significa que, para cada permutação $S_i$ onde $i$ é o índice da permutação, variando de $1$ até $n!$, ou seja, o total de permutações, o número de elementos presentes em cada permutação é sempre n. Ou seja, cada permutação $S_i$ contém $n$ elementos, mas a ordem desses elementos é diferente nas diferentes permutações.

A pontuação $F1_{\text{micro}}(\mathbf{O})$ é calculada considerando todas as classes de forma conjunta, acumulando os verdadeiros positivos ($TP$), falsos positivos ($FP$) e falsos negativos ($FN$):

$$
F1_{\text{micro}}(\mathbf{O}) = \frac{2 \cdot \text{Precision}(\mathbf{O}) \cdot \text{Recall}(\mathbf{O})}{\text{Precision}(\mathbf{O}) + \text{Recall}(\mathbf{O})}
$$

dado que:

$$
\text{Precision}(\mathbf{O}) = \frac{\sum_{i=1}^n TP_i}{\sum_{i=1}^n (TP_i + FP_i)}, \quad
\text{Recall}(\mathbf{O}) = \frac{\sum_{i=1}^n TP_i}{\sum_{i=1}^n (TP_i + FN_i)}
$$

onde $TP_i$, $FP_i$ e $FN_i$ são, respectivamente, o número de verdadeiros positivos, falsos positivos e falsos negativos da classe $i$, com base nas previsões do modelo treinado segundo a ordem $\mathbf{O}$ utilizando o conjunto de validação $(X_{\text{val}}, y_{\text{val}})$.

> **Nota:** Embora a função objetivo esteja definida sobre o conjunto completo de permutações $S_n$ (com $|S_n| = 6! = 720$ para 6 classes), na prática utilizamos um algoritmo genético para realizar uma busca heurística e eficiente dentro desse espaço, sem necessidade de exploração exaustiva.

In [25]:
def genetic_algorithm(X_train, y_train, X_val, y_val) -> tuple[list, float, list]:  
    """
    Este método implementa um algoritmo genético para encontrar a melhor ordem de classes
    para uma cadeia de classificadores, maximizando a pontuação F1 (média micro).

    Args:
        X_train (sparse matrix): Dados de treinamento.
        y_train (DataFrame): Rótulos de treinamento.
        X_val (sparse matrix): Dados de validação.
        y_val (DataFrame): Rótulos de validação.

    Retorna:
        tuple: Uma tupla contendo:
            - best_individual (list): O melhor indivíduo (ordem de classes) encontrado.
            - best_fitness (float): A melhor pontuação de aptidão (F1 média micro) alcançada.
            - history (list): Histórico das melhores pontuações de aptidão por geração.
    """
    num_classes = y_train.shape[1]
    population = initialize_population(num_classes)
    best_individual = None
    best_fitness = -1
    history = []
    
    for _ in tqdm(range(GENERATIONS), desc="Algoritmo Genético"):
        # Avaliar a aptidão de cada indivíduo na população
        fitness_scores = [evaluate_individual(ind, X_train, y_train, X_val, y_val) for ind in population]
        
        # Atualizar o melhor indivíduo
        current_best_idx = np.argmax(fitness_scores)
        if fitness_scores[current_best_idx] > best_fitness:
            best_fitness = fitness_scores[current_best_idx]
            best_individual = deepcopy(population[current_best_idx])
        
        history.append(max(fitness_scores))
        
        # Criar a próxima geração
        new_population = []
        
        # Elitismo: preservar os melhores indivíduos
        elite_idx = np.argsort(fitness_scores)[-ELITISM:]
        new_population.extend(deepcopy(population[i]) for i in elite_idx)
        
        # Preencher o restante com descendentes
        while len(new_population) < POPULATION_SIZE:
            parent1 = selection(population, fitness_scores)
            parent2 = selection(population, fitness_scores)
            child = mutate(crossover(parent1, parent2))
            new_population.append(child)
        
        population = new_population
    
    return best_individual, best_fitness, history

In [ ]:
best_order_indices, best_score, history = genetic_algorithm(
    X_train_split, y_train_split, X_val, y_val
)

Algoritmo Genético:  46%|████▌     | 23/50 [06:44<07:48, 17.36s/it]

In [ ]:
history

In [ ]:
best_order_names = [label_cols[i] for i in best_order_indices]
print(f"\nBest class order found: {best_order_names}")
print(f"Best micro F1 score: {best_score:.4f}")

In [ ]:
final_model = ClassifierChain(
    LogisticRegression(class_weight='balanced', max_iter=1000),
    order=best_order_indices
)

In [ ]:
final_model.fit(X_train_split, y_train_split)

In [ ]:
y_val_pred = final_model.predict(X_val)
print("\nClassification Report with Optimized Class Order:")
print(classification_report(y_val, y_val_pred, target_names=label_cols))

In [ ]:
# Compute confusion matrices for each label
conf_matrices = multilabel_confusion_matrix(y_val, y_val_pred_chain)
num_labels = len(label_cols)

# Set up the grid layout
cols = 3  # Number of columns in the grid
rows = int(np.ceil(num_labels / cols))  # Number of rows needed

# Create a big figure
fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
fig.suptitle('Matrizes de confusão para cada rótulo', fontsize=16, y=1.02)

# Flatten axes if needed (for cases where rows=1)
if rows == 1:
    axes = axes.reshape(1, -1)

# Loop through each label and plot its confusion matrix
for i, label in enumerate(label_cols):
    row = i // cols
    col = i % cols
    ax = axes[row, col] if rows > 1 else axes[col]
    
    sns.heatmap(
        conf_matrices[i], 
        annot=True, 
        fmt='g', 
        ax=ax,
        cmap='Blues',
        cbar=False,
        annot_kws={"size": 12}
    )
    ax.set_title(label, fontsize=14)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)

# Hide empty subplots if any
for i in range(num_labels, rows * cols):
    row = i // cols
    col = i % cols
    fig.delaxes(axes[row, col] if rows > 1 else axes[col])

plt.tight_layout()
plt.show()

In [ ]:
# Compute confusion matrices for each label
conf_matrices = multilabel_confusion_matrix(y_val, y_val_pred)
num_labels = len(label_cols)

# Set up the grid layout
cols = 3  # Number of columns in the grid
rows = int(np.ceil(num_labels / cols))  # Number of rows needed

# Create a big figure
fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
fig.suptitle('Matrizes de confusão para cada rótulo AC+CC', fontsize=16, y=1.02)

# Flatten axes if needed (for cases where rows=1)
if rows == 1:
    axes = axes.reshape(1, -1)

# Loop through each label and plot its confusion matrix
for i, label in enumerate(label_cols):
    row = i // cols
    col = i % cols
    ax = axes[row, col] if rows > 1 else axes[col]
    
    sns.heatmap(
        conf_matrices[i], 
        annot=True, 
        fmt='g', 
        ax=ax,
        cmap='Blues',
        cbar=False,
        annot_kws={"size": 12}
    )
    ax.set_title(label, fontsize=14)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)

# Hide empty subplots if any
for i in range(num_labels, rows * cols):
    row = i // cols
    col = i % cols
    fig.delaxes(axes[row, col] if rows > 1 else axes[col])

plt.tight_layout()
plt.show()

In [ ]:
def overall_metric_comparison(y_true, y_pred_cc, y_pred_ga, title_suffix=''):
    metrics = {
        "F1 Micro": [f1_score(y_true, y_pred_cc, average="micro"),
                     f1_score(y_true, y_pred_ga, average="micro")],
        "F1 Macro": [f1_score(y_true, y_pred_cc, average="macro"),
                     f1_score(y_true, y_pred_ga, average="macro")],
        "Precision": [precision_score(y_true, y_pred_cc, average="macro"),
                      precision_score(y_true, y_pred_ga, average="macro")],
        "Recall": [recall_score(y_true, y_pred_cc, average="macro"),
                   recall_score(y_true, y_pred_ga, average="macro")]
    }

    df_metrics = pd.DataFrame(metrics, index=["CC", "GA+CC"]).T
    df_metrics.plot(kind="bar", figsize=(10, 6), colormap="coolwarm")
    plt.title(f"Comparação geral de métricas {title_suffix}")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.xticks(rotation=45)
    plt.grid(False)
    plt.tight_layout()
    plt.show()

def labelwise_f1_comparison(y_true, y_pred_cc, y_pred_ga, labels):
    f1_cc = f1_score(y_true, y_pred_cc, average=None)
    f1_ga = f1_score(y_true, y_pred_ga, average=None)

    x = np.arange(len(labels))
    width = 0.35

    plt.figure(figsize=(14, 6))
    plt.bar(x - width/2, f1_cc, width, label="CC", color='skyblue')
    plt.bar(x + width/2, f1_ga, width, label="GA+CC", color='salmon')

    plt.xticks(x, labels, rotation=45)
    plt.ylabel("F1 Score")
    plt.title("Comparação de pontuação F1 por rótulo")
    plt.legend()
    plt.tight_layout()
    plt.grid(False)
    plt.show()


In [ ]:
overall_metric_comparison(y_val, y_val_pred_chain, y_val_pred, title_suffix="")
labelwise_f1_comparison(y_val, y_val_pred_chain, y_val_pred, label_cols)